# Diamond Price Prediction — Model Training

This notebook prepares the data, trains a KNN regression pipeline, evaluates it, and saves the complete pipeline for the Streamlit app.

**Step 1 — Import libraries and load the dataset**

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pickle

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load the dataset from the project's data folder
DATA_PATH = Path("../data/diamonds.csv")
df = pd.read_csv(DATA_PATH)

print("Dataset shape before cleaning:", df.shape)
print(df.head())

Dataset shape before cleaning: (53940, 11)
   Unnamed: 0  carat      cut color clarity  depth  table  price     x     y  \
0           1   0.23    Ideal     E     SI2   61.5   55.0    326  3.95  3.98   
1           2   0.21  Premium     E     SI1   59.8   61.0    326  3.89  3.84   
2           3   0.23     Good     E     VS1   56.9   65.0    327  4.05  4.07   
3           4   0.29  Premium     I     VS2   62.4   58.0    334  4.20  4.23   
4           5   0.31     Good     J     SI2   63.3   58.0    335  4.34  4.35   

      z  
0  2.43  
1  2.31  
2  2.31  
3  2.63  
4  2.75  


**Step 2 — Remove the unnecessary index column**

`Unnamed: 0`, if present, is usually an exported row index rather than a meaningful diamond feature. Remove it before training.

In [2]:
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

print("Dataset shape after cleaning:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape after cleaning: (53940, 10)
Columns: ['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'price', 'x', 'y', 'z']


**Step 3 — Define input features (X) and target (y)**

In [3]:
X = df.drop(columns=["price"])
y = df["price"]

print("Input features shape:", X.shape)
print("Target shape:", y.shape)
print("Input feature columns:", X.columns.tolist())

Input features shape: (53940, 9)
Target shape: (53940,)
Input feature columns: ['carat', 'cut', 'color', 'clarity', 'depth', 'table', 'x', 'y', 'z']


**Step 4 — Split the dataset into training and testing sets**

We use 75% of the data for training and 25% for testing. `random_state=42` makes the split reproducible.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (40455, 9)
X_test shape: (13485, 9)
y_train shape: (40455,)
y_test shape: (13485,)


**Step 5 — Identify numerical and categorical columns**

In [5]:
numerical_cols = X_train.select_dtypes(include="number").columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "str"]).columns.tolist()

print("Numerical columns:", numerical_cols)
print("Categorical columns:", categorical_cols)

Numerical columns: ['carat', 'depth', 'table', 'x', 'y', 'z']
Categorical columns: ['cut', 'color', 'clarity']


**Step 6 — Create preprocessing pipelines**

Numerical features are scaled with `StandardScaler`. Categorical features are converted into numeric columns using `OneHotEncoder`. Putting preprocessing inside a pipeline helps prevent data leakage.

In [6]:
numerical_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numerical", numerical_pipeline, numerical_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Preprocessor created successfully.")

Preprocessor created successfully.


**Step 7 — Build and train the KNN regression pipeline**

In [7]:
knn_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsRegressor(n_neighbors=5))
])

knn_pipeline.fit(X_train, y_train)

print("KNN model trained successfully.")

KNN model trained successfully.


**Step 8 — Predict prices on the test set**

In [8]:
y_pred = knn_pipeline.predict(X_test)

print("First 10 actual prices:")
print(y_test.head(10).tolist())

print("\nFirst 10 predicted prices:")
print(y_pred[:10])

First 10 actual prices:
[559, 2201, 1238, 1304, 6901, 3011, 1765, 1679, 2102, 4789]

First 10 predicted prices:
[ 586.4 2427.6 1083.4 1196.2 9642.2 3434.4 1776.8 1829.4 2390.6 6291. ]


**Step 9 — Evaluate the model**

MAE and RMSE are prediction-error measures, so lower is generally better. R² measures how much variation in the target is explained by the model; closer to 1 is generally better.

In [9]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("KNN Model Evaluation")
print("--------------------")
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))
print("R2 Score:", round(r2, 4))

KNN Model Evaluation
--------------------
MAE: 408.17
RMSE: 785.87
R2 Score: 0.9607


**Step 10 — Save the trained pipeline**

Run this cell after the model has trained successfully. The pipeline includes the fitted preprocessing steps and KNN model.

In [10]:
# Resolve paths relative to this notebook's folder
model_folder = Path("../models")
model_folder.mkdir(parents=True, exist_ok=True)

model_path = model_folder / "knn_diamond_model.pkl"

with open(model_path, "wb") as file:
    pickle.dump(knn_pipeline, file)

print("Model saved successfully!")
print("Saved location:", model_path.resolve())

Model saved successfully!
Saved location: E:\Innomatics\Diamond_Price_Prediction\models\knn_diamond_model.pkl
